In [ ]:
#| hide

# Existing Drivers

> Production-ready ROS2 camera drivers — when to use them instead of rolling your own.

## When to Use Existing Drivers

The notebooks in this project teach you how to build camera nodes from scratch. But in production, you should almost always use an existing driver first.

**Use an existing driver when:**

- Your camera has a well-supported ROS2 driver
- You need features like hardware sync, trigger, or multi-camera
- You want zero-maintenance, community-tested code

**Roll your own when:**

- No driver exists for your hardware
- You need custom preprocessing or pipeline integration
- You're learning (like in this project)

## V4L2 Camera

[`v4l2_camera`](https://github.com/ros-drivers/v4l2_camera) — the standard V4L2 driver for Linux USB cameras.

| Feature | Detail |
|---------|--------|
| Install | `sudo apt install ros-$ROS_DISTRO-v4l2-camera` |
| Run | `ros2 run v4l2_camera v4l2_camera_node` |
| Topics | `/image_raw`, `/camera_info` |
| Params | `video_device`, `pixel_format`, `image_size`, `frame_rate` |
| License | BSD |
| Maintained | Yes — part of `ros-drivers` org |

**Pros:**

- Works with any V4L2-compatible USB camera
- Handles `CameraInfo` publishing automatically
- Supports multiple pixel formats (YUYV, MJPG, etc.)

**Cons:**

- C++ only (no Python API)
- Limited to V4L2 devices (no network cameras)
- No built-in compression (use `image_transport` for that)

## USB Cam

[`usb_cam`](https://github.com/ros-drivers/usb_cam) — another popular USB camera driver.

| Feature | Detail |
|---------|--------|
| Install | `sudo apt install ros-$ROS_DISTRO-usb-cam` |
| Run | `ros2 run usb_cam usb_cam_node_exe` |
| Topics | `/image_raw`, `/camera_info` |
| Params | `video_device`, `pixel_format`, `image_width`, `image_height`, `framerate` |
| License | BSD |
| Maintained | Yes — active development |

**Pros:**

- Python and C++ support
- Works with network cameras (RTSP, HTTP)
- More flexible parameter handling

**Cons:**

- Slightly more complex configuration
- Some edge cases with certain USB chipsets

## Vendor SDKs

Many industrial camera manufacturers provide their own ROS2 drivers:

| Vendor | Package | Notes |
|--------|---------|-------|
| **FLIR / Teledyne** | `spinnaker_camera_driver` | Blackfly, Grasshopper, etc. || **Basler** | `basler_ros2` | ace2, dart cameras |
| **Allied Vision** | `allied_vision_ros2` | Alvium, Mako cameras |
| **Intel RealSense** | `realsense-ros` | D400, L500, SR300 series |
| **Stereolabs ZED** | `zed-ros2-wrapper` | ZED, ZED 2, ZED Mini |
| **Ximea** | `ximea_ros2` | xiQ, xiC cameras |

These drivers typically provide:

- Hardware-specific features (trigger, sync, exposure control)
- Calibration tools
- Multi-camera support
- Optimized for their specific hardware

## Comparison Table

| Driver | Camera types | Setup complexity | Maintenance | Multi-cam | Compression |
|--------|-------------|------------------|-------------|-----------|-------------|
| `v4l2_camera` | USB (V4L2) | Low | Good | Manual | Via `image_transport` |
| `usb_cam` | USB + network | Medium | Active | Manual | Via `image_transport` |
| `spinnaker` | FLIR only | High | Vendor | Yes | Hardware |
| `realsense` | RealSense only | Medium | Intel | Yes | Built-in |
| `zed` | ZED only | Medium | Stereolabs | Yes | Built-in |

## Quick Start: v4l2_camera

```sh
# Install
$ sudo apt install ros-$ROS_DISTRO-v4l2-camera

# Run with defaults
$ ros2 run v4l2_camera v4l2_camera_node

# Run with specific device and format
$ ros2 run v4l2_camera v4l2_camera_node --ros-args \
    -p video_device:=/dev/video0 \
    -p pixel_format:="mjpeg" \
    -p image_size:=[640,480] \
    -p frame_rate:=30.0
```

View the output:

```sh
$ ros2 run rqt_image_view rqt_image_view /image_raw
```

## Quick Start: usb_cam

```sh
# Install
$ sudo apt install ros-$ROS_DISTRO-usb-cam

# Run
$ ros2 run usb_cam usb_cam_node_exe --ros-args \
    -p video_device:=/dev/video0 \
    -p pixel_format:="mjpeg" \
    -p image_width:=640 \
    -p image_height:=480 \
    -p framerate:=30.0
```

## Decision Flowchart

```
Do you have a specific camera vendor?
├── Yes → Use vendor's ROS2 driver
└── No → Is it a USB camera?
    ├── Yes → Try v4l2_camera first
    └── No → Is it a network camera (RTSP/HTTP)?
        ├── Yes → Try usb_cam or write your own
        └── No → Check if a driver exists for your protocol
```

> **Rule of thumb:** Always try an existing driver first. If it doesn't work or doesn't have the features you need, then — and only then — build your own using the patterns from this project.